# Multi-zone CUDA fault — focused reproducer

On an A100 (Colab, torch 2.11+cu128, Python 3.13) every multi-zone single-shooting case died with
`CUDA error: an illegal memory access was encountered`; 1-zone cases passed, and the same 10-zone cases pass on
the laptop (RTX PRO 2000, torch 2.13+cu130, Python 3.12).

**Status (2026-09-08).** Reproduced in this notebook with `MODE="full"`, 10 zones, `cuda_graph`, `MAXITER=3`:
the fault surfaces at the synchronize right after the *first* bundle call, i.e. inside the capture routine
(one eager warmup, the graph record, one replay, a replay-vs-eager parity check). It reproduced with the fixed
memory sampler, so the sampler thread is **cleared**. Remaining hypotheses, and the switch that tests each:

| Hypothesis | Switch | Expected if true |
|---|---|---|
| torch 2.11 CUDA-graph bug (fixed by 2.13) | `TORCH_SPEC = "torch==2.13.0 --index-url https://download.pytorch.org/whl/cu128"` | passes |
| fault needs graph capture/replay | `BACKEND = "eager"` | passes |
| which capture phase launches it | `SYNC_PHASES = True` (default) | traceback note says `warmup` / `record` / `replay` / `parity-check` |

`SYNC_PHASES` sets `T4B_CUDA_GRAPH_SYNC_PHASES=1`: the wrapper synchronizes after every capture phase so the
asynchronous fault is attributed to the phase that launched it, and it adds a note to the exception.

`CUDA_LAUNCH_BLOCKING` makes kernel launches synchronous. That is **incompatible with stream capture** (the
record phase will fail with a different error), so use it only together with `BACKEND = "eager"` if eager
also faults, to name the kernel.

* `MODE` `smoke` (2 h horizon, 1 iteration, minutes) vs `full` (120 h, 300 iterations); `MAXITER` caps the
  iterations independently of the mode (0 = mode default). The crashing case is `full` / 10 zones / `cuda_graph`.
* `derivative_stats` in the output shows `captures` and `replays`; a run with `replays: 0` never executed the
  captured graph outside the capture routine's own parity replay.

Run the cells top to bottom. Cell 1 must run **before** anything imports torch. Restart the runtime after a
crash (the CUDA context is poisoned) and after changing `TORCH_SPEC`.

In [ ]:
#@title 1. Configuration (runs before torch is imported) { display-mode: "form" }
import os
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"  #@param {type:"string"}
GIT_REF = "feature/issue-128/collocation-initialization"       #@param {type:"string"}
N_ZONES = 10                    #@param {type:"integer"}
SOLVER = "slsqp-single-shooting"  #@param ["slsqp-single-shooting", "custom-batched-sqp", "ipopt-collocation"]
N_STARTS = 1                    #@param {type:"integer"}
BACKEND = "cuda_graph"          #@param ["cuda_graph", "eager"]
MODE = "smoke"                  #@param ["smoke", "full"]
MAXITER = 0                     #@param {type:"integer"}
#@markdown `MAXITER` 0 = the mode's default (smoke: 1, full: 300). Use `MODE="full"` with `MAXITER=3` to
#@markdown capture the real 120 h / 10-zone graph and replay it a few times without waiting for 300 iterations.
MEMORY_SAMPLER = "fixed"        #@param ["off", "fixed", "legacy_mem_get_info"]
SYNC_PHASES = True              #@param {type:"boolean"}
CUDA_LAUNCH_BLOCKING = False    #@param {type:"boolean"}
TORCH_SPEC = ""                 #@param {type:"string"}
#@markdown `TORCH_SPEC` empty = keep the runtime's torch. Example to match the laptop that passes:
#@markdown `torch==2.13.0 --index-url https://download.pytorch.org/whl/cu128` (restart the runtime after changing it).
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" if CUDA_LAUNCH_BLOCKING else "0"
os.environ["T4B_CUDA_GRAPH_SYNC_PHASES"] = "1" if SYNC_PHASES else "0"
os.environ["T4B_BENCHMARK_MODE"] = MODE
print({k: v for k, v in globals().items() if k in ("N_ZONES", "SOLVER", "N_STARTS", "BACKEND", "MODE", "MAXITER", "MEMORY_SAMPLER", "SYNC_PHASES", "CUDA_LAUNCH_BLOCKING", "TORCH_SPEC")})

In [ ]:
#@title 2. Install Twin4Build from the branch (same bootstrap as the benchmark notebooks)
import pathlib, subprocess, sys
REPO = pathlib.Path("/content/Twin4Build")
if "google.colab" in sys.modules:
    if not REPO.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False, capture_output=True)
    if TORCH_SPEC.strip():
        print("installing", TORCH_SPEC)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *TORCH_SPEC.split()], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "psutil"], check=True)
else:
    REPO = pathlib.Path.cwd()
    if REPO.name == "benchmarks":
        REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import torch, twin4build
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"), "T4B_CUDA_GRAPH_SYNC_PHASES =", os.environ.get("T4B_CUDA_GRAPH_SYNC_PHASES"))

In [ ]:
#@title 3. Build the case exactly as the harness does
import time, json, traceback, threading
import benchmarks.common as common
from benchmarks.common import BenchmarkConfig, batched_estimation_problem, seed_everything, _estimation_window, ESTIMATION_METHODS
import twin4build as tb

config = BenchmarkConfig(mode=MODE)
seed_everything(config.seed)
t0 = time.time()
setup = batched_estimation_problem(N_ZONES, config)
model = setup["model"]; model.to("cuda", torch.float64)
print(f"setup built in {time.time()-t0:.0f} s: {N_ZONES} zones, hours={config.hours}, maxiter={config.estimation_maxiter}")

# Optional: re-create the OLD sampler (CUDA runtime call from a side thread) to reproduce deliberately.
legacy_stop = threading.Event()
def _legacy_poll():
    while not legacy_stop.wait(1.0):
        try:
            torch.cuda.mem_get_info()
        except Exception as exc:
            print("legacy sampler error:", repr(exc)[:200])
if MEMORY_SAMPLER == "legacy_mem_get_info":
    threading.Thread(target=_legacy_poll, daemon=True, name="legacy-mem-sampler").start()
    print("legacy mem_get_info sampler thread started (1 Hz)")
elif MEMORY_SAMPLER == "off":
    common.MEMORY_SAMPLER_INTERVAL_SECONDS = 3600.0  # effectively disables the fixed sampler

In [ ]:
#@title 4. Run the case in-process (full traceback on failure)
estimator = tb.Estimator(tb.Simulator(model, execution_mode="functional", execution_backend=BACKEND))
options = {"maxiter": MAXITER or config.estimation_maxiter}
method = ESTIMATION_METHODS[SOLVER]
if SOLVER == "ipopt-collocation":
    options["hessian"] = "exact"
if SOLVER == "custom-batched-sqp":
    options.update({"n_starts": N_STARTS, "batch_size": N_STARTS, "start_seed": config.seed, "start_strategy": "local", "start_spread": 0.1})
print("method", method, "options", options, "backend", BACKEND)
torch.cuda.reset_peak_memory_stats()
t0 = time.time(); outcome = None
try:
    result, seconds = common.timed("cuda", lambda: estimator.estimate(
        parameters=setup["parameters"], measurements=setup["measurements"], method=method, options=options, **_estimation_window(config)))
    outcome = {"status": "ok", "seconds": seconds, "iterations": result.get("iterations"), "message": result.get("message"),
               "final_objective": result.get("final_objective"), "derivative_stats": result.get("derivative_stats")}
except BaseException as exc:
    chain = [exc, exc.__cause__, exc.__context__]
    notes = [n for e in chain if e is not None for n in getattr(e, "__notes__", [])]
    outcome = {"status": "FAILED", "seconds": time.time() - t0, "error": repr(exc)[:400], "capture_phase_notes": notes}
    traceback.print_exc()
finally:
    legacy_stop.set()
print(json.dumps(outcome, indent=1, default=str))
print("memory:", json.dumps({k: (round(v / 2**30, 2) if isinstance(v, int) and v > 1e6 else v) for k, v in common.memory_stats_of_last_timed().items()}, indent=1))
print("torch peak allocated GiB", round(torch.cuda.max_memory_allocated() / 2**30, 2), "reserved", round(torch.cuda.max_memory_reserved() / 2**30, 2))

### How to read the outcome

`capture_phase_notes` in the outcome (and the `CUDA graph capture phase: ...` note in the traceback) says which
phase of the first bundle call the fault surfaced in, provided `SYNC_PHASES` was on:

| Phase in the note | Meaning |
|---|---|
| `warmup` | the eager rollout itself faults on this torch build — capture is not the cause; confirm with `BACKEND="eager"` |
| `record` | a kernel misbehaves under stream capture (allocator / cuBLAS workspace / functorch under capture) |
| `replay` | the recorded graph is invalid when executed: torch-version graph bug is the prime suspect |
| `parity-check` | replay ran but the comparison faulted — unusual, paste it |

Suggested order, restarting the runtime between runs:

1. `MODE="full"`, `MAXITER=3`, `cuda_graph`, `SYNC_PHASES=True` — attributes the phase.
2. Same with `TORCH_SPEC = "torch==2.13.0 --index-url https://download.pytorch.org/whl/cu128"` — the laptop's
   torch on the A100. A pass means the benchmark notebook should pin torch and nothing in Twin4Build changes.
3. Same with `BACKEND="eager"` on the runtime's torch — a pass confines the fault to capture/replay.

Paste the outcome JSON and the traceback back.

In [ ]:
#@title 5. (Optional) the same case through the real harness subprocess, with the fixed sampler
RUN_HARNESS = False  #@param {type:"boolean"}
if RUN_HARNESS:
    from benchmarks.common import _run_estimation_case_subprocess
    row = _run_estimation_case_subprocess(config, n_zones=N_ZONES, device="cuda", solver=SOLVER, n_starts=N_STARTS, repetition=0)
    print({k: row.get(k) for k in ("status", "seconds", "iterations", "message", "error", "child_returncode")})
    if row.get("child_traceback"):
        print(row["child_traceback"][-3000:])